**Full MPNST TE Counting Run**

Run this only after the pilot has succeeded.

This notebook builds full-run scripts for all samples in the manifest

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
mpnst_root <- file.path(project_root, "05_MPNST_TE_analysis")
dir.create(mpnst_root, showWarnings = FALSE, recursive = TRUE)
setwd(mpnst_root)
manifest_dir <- file.path(mpnst_root, "manifests")
counts_dir <- file.path(mpnst_root, "te_counts")
full_run_dir <- file.path(counts_dir, "full_run")
dir.create(full_run_dir, showWarnings = FALSE, recursive = TRUE)

sample_manifest_file <- file.path(manifest_dir, "MPNST_RNAseq_sample_manifest.csv")
stopifnot(file.exists(sample_manifest_file))

manifest <- read_csv(sample_manifest_file, show_col_types = FALSE)
manifest %>% count(PRC2_status, file_type)

In [ ]:
# Choose TE counting strategy

# Using SalmonTE on paired-end FASTQ files -> quantifies repeat/transposable-element expression directly from RNA-seq reads

te_counting_strategy <- "SalmonTE"

if (!te_counting_strategy %in% c("SalmonTE", "TEcount", "skip")) {
  stop("Invalid te_counting_strategy")
}

te_counting_strategy

In [ ]:
# Build full run script


raw_data_dir <- file.path(mpnst_root, "raw_or_external_data")
software_dir <- file.path(mpnst_root, "software")
salmonte_output_dir <- file.path(full_run_dir, "SalmonTE_outputs")

dir.create(raw_data_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(software_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(salmonte_output_dir, showWarnings = FALSE, recursive = TRUE)

# Keep only FASTQ samples - GSE206527 SRA metadata is paired-end FASTQ.
run_manifest <- manifest %>%
  filter(file_type %in% c("FASTQ_PE", "FASTQ_SE")) %>%
  arrange(PRC2_status, sample_id)

if (nrow(run_manifest) == 0) {
  stop("No FASTQ samples found in the manifest. Check MPNST_RNAseq_sample_manifest.csv.")
}

# Query ENA for exact FASTQ URLs, MD5 checksums and file sizes
get_ena_fastq_record <- function(run_accession) {
  ena_url <- paste0(
    "https://www.ebi.ac.uk/ena/portal/api/filereport?",
    "accession=", run_accession,
    "&result=read_run",
    "&fields=run_accession,fastq_ftp,fastq_md5,fastq_bytes",
    "&format=tsv"
  )

  tryCatch(
    readr::read_tsv(ena_url, show_col_types = FALSE),
    error = function(e) {
      stop("ENA query failed for ", run_accession, ": ", conditionMessage(e))
    }
  )
}

message("Querying ENA for ", nrow(run_manifest), " runs...")
ena_run_records <- purrr::map_dfr(run_manifest$accession, get_ena_fastq_record)

if (nrow(ena_run_records) != nrow(run_manifest)) {
  stop("ENA did not return exactly one record for every run.")
}

# Convert ENA semicolon-separated paired FASTQ fields into one row per file
full_fastq_files <- purrr::map_dfr(seq_len(nrow(ena_run_records)), function(i) {
  record <- ena_run_records[i, ]
  ftp <- stringr::str_split(record$fastq_ftp, ";", simplify = TRUE) %>% as.character()
  md5 <- stringr::str_split(record$fastq_md5, ";", simplify = TRUE) %>% as.character()
  bytes <- stringr::str_split(record$fastq_bytes, ";", simplify = TRUE) %>% as.character()

  ftp <- ftp[nzchar(ftp)]
  md5 <- md5[nzchar(md5)]
  bytes <- bytes[nzchar(bytes)]

  if (length(ftp) == 0) {
    stop("No FASTQ links returned by ENA for ", record$run_accession)
  }

  sample_dir <- file.path(raw_data_dir, record$run_accession)

  tibble(
    sample_id = record$run_accession,
    read_number = seq_along(ftp),
    download_url = paste0("https://", ftp),
    local_path = file.path(sample_dir, basename(ftp)),
    expected_md5 = md5,
    expected_size_gb = as.numeric(bytes) / 1e9
  )
})

write_csv(
  full_fastq_files,
  file.path(full_run_dir, "MPNST_full_FASTQ_download_manifest.csv")
)

# Replace provisional FASTQ paths with exact local paths generated from ENA
full_fastq_paths <- full_fastq_files %>%
  select(sample_id, read_number, local_path) %>%
  tidyr::pivot_wider(
    names_from = read_number,
    values_from = local_path,
    names_prefix = "fastq_"
  )

full_manifest <- run_manifest %>%
  select(-any_of(c("fastq_1", "fastq_2"))) %>%
  left_join(full_fastq_paths, by = "sample_id")

write_csv(full_manifest, file.path(full_run_dir, "MPNST_full_run_manifest.csv"))

# Write a resumable download script
download_script <- file.path(full_run_dir, "run_full_FASTQ_download.sh")

download_lines <- c(
  "#!/usr/bin/env bash",
  "set -euo pipefail",
  "",
  "# Full MPNST FASTQ download from ENA.",
  "# Safe to rerun: existing files with matching MD5 checksums are skipped.",
  ""
)

for (i in seq_len(nrow(full_fastq_files))) {
  f <- full_fastq_files[i, ]
  download_lines <- c(
    download_lines,
    "",
    paste0("echo 'Checking ", basename(f$local_path), "'"),
    paste0("mkdir -p ", shQuote(dirname(f$local_path))),
    paste0(
      "if [ -s ", shQuote(f$local_path), " ] && echo ",
      shQuote(paste(f$expected_md5, f$local_path)),
      " | md5sum -c - >/dev/null 2>&1; then"
    ),
    "  echo '  already downloaded and MD5 verified'",
    "else",
    paste("  wget -c -O", shQuote(f$local_path), shQuote(f$download_url)),
    paste(
      "  echo", shQuote(paste(f$expected_md5, f$local_path)),
      "| md5sum -c -"
    ),
    "fi"
  )
}

download_lines <- c(download_lines, "", "echo 'Full FASTQ download completed successfully.'")
writeLines(download_lines, download_script)
Sys.chmod(download_script, mode = "0755")

# Locate the local SalmonTE installation
local_salmonte_repo_script <- file.path(software_dir, "SalmonTE", "SalmonTE.py")
local_salmonte_python <- file.path(software_dir, "salmonte_venv", "bin", "python")

if (file.exists(local_salmonte_repo_script) && file.exists(local_salmonte_python)) {
  salmonte_prefix <- paste(shQuote(local_salmonte_python), shQuote(local_salmonte_repo_script))
  salmonte_mode <- "local virtual environment"
} else {
  stop(
    "Local SalmonTE is not configured. Rerun Notebook 25.2 Cell 3 and the install script before running the full analysis."
  )
}

cat("SalmonTE mode:", salmonte_mode, "\n")

# The full run intentionally reruns the four pilot samples

# Write the full SalmonTE script
full_script <- file.path(full_run_dir, "run_full_SalmonTE.sh")

salmonte_lines <- c(
  "#!/usr/bin/env bash",
  "set -euo pipefail",
  "",
  "# Full SalmonTE run for MPNST RNA-seq samples.",
  "# Safe to rerun: samples with EXPR.csv already present are skipped.",
  "# --exprtype=count requests count output for differential expression analysis.",
  "",
  paste0("mkdir -p ", shQuote(salmonte_output_dir))
)

for (i in seq_len(nrow(full_manifest))) {
  s <- full_manifest[i, ]
  out_i <- file.path(salmonte_output_dir, s$sample_id)
  expr_i <- file.path(out_i, "EXPR.csv")

  if (s$file_type == "FASTQ_PE") {
    command_i <- paste(
      salmonte_prefix,
      "quant --reference=hs --exprtype=count --outpath", shQuote(out_i),
      shQuote(s$fastq_1),
      shQuote(s$fastq_2)
    )
  } else {
    command_i <- paste(
      salmonte_prefix,
      "quant --reference=hs --exprtype=count --outpath", shQuote(out_i),
      shQuote(s$fastq_1)
    )
  }

  salmonte_lines <- c(
    salmonte_lines,
    "",
    paste0("echo 'Checking SalmonTE output for ", s$sample_id, "'"),
    paste0("mkdir -p ", shQuote(out_i)),
    paste0("if [ -s ", shQuote(expr_i), " ]; then"),
    "  echo '  EXPR.csv already exists; skipping'",
    "else",
    paste0("  echo '  Running SalmonTE for ", s$sample_id, "'"),
    paste0("  ", command_i),
    "fi"
  )
}

writeLines(salmonte_lines, full_script)
Sys.chmod(full_script, mode = "0755")

cat("Samples in full run:", nrow(full_manifest), "\n")
cat("FASTQ files:", nrow(full_fastq_files), "\n")
cat("Expected compressed FASTQ size:", round(sum(full_fastq_files$expected_size_gb), 2), "GB\n")
cat("Download script:\n", download_script, "\n")
cat("SalmonTE script:\n", full_script, "\n")

list(
  download_script = download_script,
  salmonte_script = full_script,
  full_manifest = file.path(full_run_dir, "MPNST_full_run_manifest.csv"),
  fastq_manifest = file.path(full_run_dir, "MPNST_full_FASTQ_download_manifest.csv")
)

In [ ]:
# Run full script from terminal


# Step 1: download all FASTQ files
cat("bash te_counts/full_run/run_full_FASTQ_download.sh\n")

# Step 2: run SalmonTE for all samples
cat("bash te_counts/full_run/run_full_SalmonTE.sh\n")


In [ ]:
# Combine TE outputs

combined_counts_file <- file.path(counts_dir, "MPNST_TE_counts_matrix.csv")
salmonte_output_dir <- file.path(full_run_dir, "SalmonTE_outputs")
full_manifest_file <- file.path(full_run_dir, "MPNST_full_run_manifest.csv")

if (!file.exists(full_manifest_file)) {
  stop("Missing full-run manifest. Rerun Cell 3 first.")
}

full_manifest <- read_csv(full_manifest_file, show_col_types = FALSE)
expected_samples <- full_manifest$sample_id

expr_files <- list.files(
  salmonte_output_dir,
  pattern = "^EXPR\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

cat("EXPR.csv files found:", length(expr_files), "\n")
cat("Expected samples:", length(expected_samples), "\n")

found_samples <- basename(dirname(expr_files))
missing_samples <- setdiff(expected_samples, found_samples)

if (length(missing_samples) > 0) {
  print(tibble(missing_sample = missing_samples))
  stop("Some SalmonTE outputs are missing. Rerun the full SalmonTE script before combining counts.")
}

# SalmonTE versions can differ slightly in column names
read_salmonte_expr <- function(path) {
  sample_id <- basename(dirname(path))
  x <- read_csv(path, show_col_types = FALSE)

  feature_candidates <- c("TE", "Name", "feature_id", "target_id", "Transcript", "transcript")
  feature_col <- intersect(feature_candidates, colnames(x))[1]
  if (is.na(feature_col)) {
    feature_col <- colnames(x)[1]
  }

  count_candidates <- c("NumReads", "count", "counts", "Count", "read_count", sample_id)
  count_col <- intersect(count_candidates, colnames(x))[1]

  if (is.na(count_col)) {
    numeric_cols <- colnames(x)[vapply(x, is.numeric, logical(1))]
    numeric_cols <- setdiff(numeric_cols, feature_col)
    if (length(numeric_cols) == 0) {
      stop("Could not identify a count column in ", path)
    }
    count_col <- numeric_cols[length(numeric_cols)]
  }

  x %>%
    transmute(
      feature_id = as.character(.data[[feature_col]]),
      !!sample_id := as.numeric(.data[[count_col]])
    )
}

count_tables <- purrr::map(expr_files, read_salmonte_expr)
count_matrix <- purrr::reduce(count_tables, full_join, by = "feature_id") %>%
  arrange(feature_id)

# Reorder columns to match the sample manifest
count_matrix <- count_matrix %>%
  select(feature_id, all_of(expected_samples))

write_csv(count_matrix, combined_counts_file)

cat("Combined count matrix written to:\n", combined_counts_file, "\n")
cat("TE features:", nrow(count_matrix), "\n")
cat("Samples:", ncol(count_matrix) - 1, "\n")

# Summary for records
completion_summary <- tibble(
  expected_samples = length(expected_samples),
  expr_files_found = length(expr_files),
  te_features = nrow(count_matrix),
  combined_counts_file = combined_counts_file
)

write_csv(
  completion_summary,
  file.path(full_run_dir, "MPNST_full_SalmonTE_completion_summary.csv")
)

completion_summary